## Here is a short script to generate text lists to control the scan parameters

In [86]:
import os
import sys

if '../' not in sys.path:
    sys.path.append('../')

In [87]:
import numpy as np

In [88]:
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

In [89]:
masses_low = np.geomspace(0.1, 100)
masses_high = np.geomspace(100, 1000)

masses = np.concatenate(
    (masses_low, masses_high[1:]) 
)

## define the scan range for m_de

In [90]:
mass_ratios = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 25.0, 50.0, 100.0, 1000.0]) 

num_mdp_per_mde = len(mass_ratios)

m_de_range = masses_low
m_dp_ranges = np.zeros((len(m_de_range), num_mdp_per_mde))


In [95]:
for (i, m_de) in enumerate(m_de_range):
    for j in range(num_mdp_per_mde):
        m_dp_ranges[i, j] = m_de*mass_ratios[j]

rate_masses = m_dp_ranges.flatten()

In [98]:
coulomb_mass_list_name = 'coulomb_mass_list.txt'
cml_path = os.path.join('../input/', coulomb_mass_list_name)

with open(cml_path, 'w') as out_file:
    for m in rate_masses:
        print(f'{m:.3e}', file=out_file)

In [99]:
ann_mass_list_name = 'annihilation_masses.csv'
aml_path = os.path.join('../input/', ann_mass_list_name)

with open(aml_path, 'w') as out_file:
    for m in rate_masses:
        print(f'{m:.3e}, {1e-2*m:.3e}, {1e3*m:.3e}', file=out_file)

In [121]:
file_txt = open('/Users/duncan/scrap/file_list', 'r')
files_run_so_far = file_txt.readlines()[1:]

In [135]:
float(files_run_so_far[0].split('_')[4])

0.1151

In [138]:
mass_points_run_so_far = np.zeros(len(files_run_so_far))

for (i, fname) in enumerate(files_run_so_far):
    mass_points_run_so_far[i]= float(fname.split('_')[4])

mass_points_run_so_far.sort()

In [146]:
mass_points_run_so_far

array([1.000e-01, 1.151e-01, 1.326e-01, 1.526e-01, 1.758e-01, 2.000e-01,
       2.024e-01, 2.303e-01, 2.330e-01, 2.651e-01, 2.683e-01, 3.000e-01,
       3.053e-01, 3.089e-01, 3.454e-01, 3.515e-01, 3.556e-01, 3.977e-01,
       4.000e-01, 4.047e-01, 4.095e-01, 4.579e-01, 4.606e-01, 4.660e-01,
       4.715e-01, 5.000e-01, 5.273e-01, 5.303e-01, 5.365e-01, 5.429e-01,
       5.757e-01, 6.000e-01, 6.071e-01, 6.106e-01, 6.178e-01, 6.251e-01,
       6.629e-01, 6.908e-01, 6.990e-01, 7.000e-01, 7.030e-01, 7.113e-01,
       7.197e-01, 7.632e-01, 7.954e-01, 8.000e-01, 8.048e-01, 8.060e-01,
       8.094e-01, 8.190e-01, 8.286e-01, 8.788e-01, 9.000e-01, 9.159e-01,
       9.211e-01, 9.267e-01, 9.280e-01, 9.320e-01, 9.430e-01, 9.541e-01,
       1.000e+00, 1.012e+00, 1.036e+00, 1.055e+00, 1.061e+00, 1.067e+00,
       1.068e+00, 1.073e+00, 1.086e+00, 1.099e+00, 1.151e+00, 1.165e+00,
       1.193e+00, 1.214e+00, 1.221e+00, 1.228e+00, 1.230e+00, 1.236e+00,
       1.250e+00, 1.265e+00, 1.326e+00, 1.341e+00, 

In [161]:
rate_masses_temp = np.zeros(len(rate_masses))
for (i, rm) in enumerate(rate_masses):
    rate_masses_temp[i] = float(f'{rm:.3e}')

In [168]:
np.unique(rate_masses_temp).shape

(652,)

In [164]:
np.intersect1d(rate_masses_temp, mass_points_run_so_far).shape

(631,)

In [166]:
np.setdiff1d(rate_masses_temp, mass_points_run_so_far)

array([   7.906,    9.103,   15.81 ,   18.21 ,   23.72 ,   39.53 ,
         41.2  ,   47.44 ,   48.07 ,   55.34 ,   61.8  ,   63.25 ,
         68.66 ,   71.15 ,   79.06 ,  138.9  ,  343.3  ,  686.6  ,
        790.6  , 6866.   , 7906.   ])

In [102]:
adm_scan_file = 'adm_Neff_scan.txt'
adms_path = os.path.join('../input/', adm_scan_file)

eps_min = 1e-10
eps_max = 1e-4

num_eps = 50


# heres how to make all the parameter points as a flattened array
N_param_points = len(m_de_range)*num_mdp_per_mde*num_eps

print(N_param_points)

flattened_scan = np.zeros((N_param_points, 3))

index = 0

for (i, m_de) in enumerate(m_de_range):
    for j in range(num_mdp_per_mde):
        m_dp = m_dp_ranges[i,j]
        for (k, eps) in enumerate(np.geomspace(eps_min, eps_max, num_eps)):
            flattened_scan[index, 0] = m_de
            flattened_scan[index, 1] = m_dp
            flattened_scan[index, 2] = eps
            index += 1


with open(adms_path, 'w') as out_file:
    np.savetxt(out_file, flattened_scan, fmt='%.3e', delimiter=',', header='m_de [MeV], m_dp [MeV], epsilon')


35000


In [103]:
flattened_scan[0, 0]

0.1

In [80]:
m_de_range[0]

0.1

## Check for corrupted npz files

In [174]:
check_dir = '../output/rates/coulomb/cluster/scan/'

In [175]:
bad_files = []
for f in os.listdir(check_dir):
    if not f.endswith('.npz'):
        continue
    test = np.load(os.path.join(check_dir, f))
    key_0 = list(test.keys())[0]
    
    try:
        foo = test[key_0]

    except:
        bad_files.append(f)

In [176]:
bad_files

[]

In [177]:
for bad_file in bad_files:
    os.remove(os.path.join(check_dir, bad_file))